In [8]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

"""
 1,load data from data/processed/train_raw.csv
 2, define preprocess:  

      删除无关列
      缺失值填补
      类别变量编码
      连续特征归一化或标准化
3， 对处理后的数据做一次简单的断言测试：
      没有确实值
      特征数量符合预取
4，将输出保存到processed/train_feat.csv
5, commit

"""

In [20]:
# ── 2. 预处理函数 ─────────────────────────────────────────
def preprocess(df, scale=True):
    """
    完整预处理流程：
    - 删除无关列
    - 缺失值填补
    - 类别变量 One-Hot 编码
    - 连续特征标准化（可选）
    """
    df = df.copy()

    # 2-1 删除无关列
    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    df.drop(columns=drop_cols, inplace=True)
    print(f"\n[Step1] 删除无关列后维度: {df.shape}")

    # 2-2 缺失值填补
    age_imputer = SimpleImputer(strategy='median')
    df['Age'] = age_imputer.fit_transform(df[['Age']]).ravel()

    emb_imputer = SimpleImputer(strategy='most_frequent')
    df['Embarked'] = emb_imputer.fit_transform(df[['Embarked']]).ravel()

    print(f"[Step2] 填补后缺失值总数: {df.isnull().sum().sum()}")

    # 2-3 One-Hot 编码
    df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)
    print(f"[Step3] 编码后维度: {df.shape}")
    print(f"        当前列名: {df.columns.tolist()}")

    # 将 bool 列转为 int（True→1, False→0） 
    bool_cols = df.select_dtypes(include='bool').columns 
    df[bool_cols] = df[bool_cols].astype(int)

    # 2-4 连续特征标准化（可选）
    if scale:
        scale_cols = ['Age', 'Fare', 'SibSp', 'Parch']
        scaler = StandardScaler()
        df[scale_cols] = scaler.fit_transform(df[scale_cols])
        print(f"[Step4] 已标准化列: {scale_cols}")

    return df



In [21]:
# ── 1. 加载数据 ──────────────────────────────────────────
df = pd.read_csv('../data/processed/train_raw.csv')
print(f"原始数据维度: {df.shape}")
print(f"原始列名: {df.columns.tolist()}")


原始数据维度: (891, 12)
原始列名: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [22]:
# ── 3. 执行预处理 ─────────────────────────────────────────
df_feat = preprocess(df, scale=True)
print(f"\n处理后数据维度: {df_feat.shape}")
print(df_feat.head())


[Step1] 删除无关列后维度: (891, 8)
[Step2] 填补后缺失值总数: 0
[Step3] 编码后维度: (891, 9)
        当前列名: ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']
[Step4] 已标准化列: ['Age', 'Fare', 'SibSp', 'Parch']

处理后数据维度: (891, 9)
   Survived  Pclass       Age     SibSp     Parch      Fare  Sex_male  \
0         0       3 -0.565736  0.432793 -0.473674 -0.502445         1   
1         1       1  0.663861  0.432793 -0.473674  0.786845         0   
2         1       3 -0.258337 -0.474545 -0.473674 -0.488854         0   
3         1       1  0.433312  0.432793 -0.473674  0.420730         0   
4         0       3  0.433312 -0.474545 -0.473674 -0.486337         1   

   Embarked_Q  Embarked_S  
0           0           1  
1           0           0  
2           0           1  
3           0           1  
4           0           1  


In [24]:
print(df_feat.shape, df_feat.columns.tolist())

(891, 9) ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']


In [25]:
# ── 4. 断言测试 ───────────────────────────────────────────
print("\n── 断言测试 ──")

# 测试1：无缺失值
assert df_feat.isnull().sum().sum() == 0, \
    "[FAIL] 存在缺失值"
print("[PASS] 无缺失值")

# 测试2：特征数量符合预期
#  原始12列
# - 删除4列（PassengerId, Name, Ticket, Cabin）= 8列
# - Sex: 1列 → 1列（drop_first去掉Female）
# - Embarked: 1列 → 2列（drop_first去掉C，保留Q/S）
# 合计: 4(数值) + 1(Sex_male) + 2(Embarked_Q/S) + 1(Survived) + 1(Pclass) = 9列
EXPECTED_COLS = 9
EXPECTED_COLS = 9
assert df_feat.shape[1] == EXPECTED_COLS, \
    f"[FAIL] 特征数量不符：期望 {EXPECTED_COLS}，实际 {df_feat.shape[1]}"
print(f"[PASS] 特征数量正确：{df_feat.shape[1]} 列")

# 测试3：目标列存在且值域正确
assert set(df_feat['Survived'].unique()).issubset({0, 1}), \
    "[FAIL] Survived 列值域异常"
print("[PASS] Survived 列值域正常 {0, 1}")


# 测试4：行数不变
assert len(df_feat) == len(df), \
    f"[FAIL] 行数变化：原 {len(df)} 行，现 {len(df_feat)} 行"
print(f"[PASS] 行数不变：{len(df_feat)} 行")


── 断言测试 ──
[PASS] 无缺失值
[PASS] 特征数量正确：9 列
[PASS] Survived 列值域正常 {0, 1}
[PASS] 行数不变：891 行


In [28]:
# ── 5. 保存结果 ───────────────────────────────────────────
import os
os.makedirs('../data/processed', exist_ok=True)
df_feat.to_csv('../data/processed/train_feat.csv', index=False)
print("\n✅ 已保存：data/processed/train_feat.csv")
print(f"   最终维度：{df_feat.shape}")


✅ 已保存：data/processed/train_feat.csv
   最终维度：(891, 9)
